# Прогноз цены автомобиля: анализ и готовая модель

Цель проекта - построить модель для оценки цены автомобиля и объяснить, какие признаки сильнее всего влияют на прогноз.

В ноутбуке есть полный цикл: загрузка данных, EDA, графики, baseline, финальная модель, метрики, анализ ошибок, важность признаков и сохранение модели. Если CSV с реальными данными не найден, создается демонстрационный датасет, чтобы ноутбук можно было выполнить сразу.

## 1. Импорт библиотек

Если библиотеки не установлены, выполните: `%pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    import joblib
except ImportError:
    joblib = None

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 120)

RANDOM_STATE = 42
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

## 2. Загрузка данных

Положите CSV-файл в корень репозитория или укажите путь в `DATA_PATH`. Поддерживаются разные названия целевой колонки: `price`, `selling_price`, `Price`, `Цена`.

In [ ]:
DATA_PATH = None  # например: 'data/cars.csv'

def make_demo_car_dataset(n=700, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    brands = np.array(['Toyota', 'BMW', 'Mercedes', 'Hyundai', 'Kia', 'Ford', 'Lada', 'Volkswagen'])
    fuel_types = np.array(['Petrol', 'Diesel', 'Hybrid', 'Electric'])
    transmissions = np.array(['Manual', 'Automatic'])
    body_types = np.array(['Sedan', 'SUV', 'Hatchback', 'Wagon', 'Coupe'])

    brand = rng.choice(brands, n, p=[.18, .10, .08, .17, .14, .12, .11, .10])
    fuel = rng.choice(fuel_types, n, p=[.52, .24, .16, .08])
    transmission = rng.choice(transmissions, n, p=[.46, .54])
    body = rng.choice(body_types, n, p=[.34, .30, .20, .10, .06])
    year = rng.integers(2005, 2025, n)
    mileage = np.maximum(0, rng.normal((2026 - year) * 14000, 22000)).astype(int)
    engine_volume = np.round(rng.normal(1.8, 0.55, n).clip(0.8, 4.5), 1)
    horsepower = np.round((engine_volume * rng.normal(88, 16, n)).clip(65, 420)).astype(int)
    owners = rng.choice([1, 2, 3, 4], n, p=[.46, .34, .15, .05])

    brand_effect = {'BMW': 9000, 'Mercedes': 10500, 'Toyota': 5200, 'Volkswagen': 3800, 'Hyundai': 1800, 'Kia': 1600, 'Ford': 900, 'Lada': -2600}
    fuel_effect = {'Petrol': 0, 'Diesel': 1200, 'Hybrid': 4200, 'Electric': 7600}
    transmission_effect = {'Manual': 0, 'Automatic': 2800}
    body_effect = {'Sedan': 0, 'SUV': 4300, 'Hatchback': -900, 'Wagon': 600, 'Coupe': 3100}

    price = (4500 + (year - 2005) * 1150 - mileage * 0.055 + horsepower * 85 + engine_volume * 1700 - owners * 950 + pd.Series(brand).map(brand_effect).to_numpy() + pd.Series(fuel).map(fuel_effect).to_numpy() + pd.Series(transmission).map(transmission_effect).to_numpy() + pd.Series(body).map(body_effect).to_numpy() + rng.normal(0, 3200, n))
    price = np.maximum(price, 1500).round(0)

    df_demo = pd.DataFrame({'brand': brand, 'model_year': year, 'mileage': mileage, 'fuel_type': fuel, 'transmission': transmission, 'body_type': body, 'engine_volume': engine_volume, 'horsepower': horsepower, 'owners': owners, 'price': price})
    for col in ['engine_volume', 'horsepower', 'fuel_type']:
        miss_idx = rng.choice(df_demo.index, size=int(n * 0.03), replace=False)
        df_demo.loc[miss_idx, col] = np.nan
    return df_demo

def find_dataset(data_path=None):
    if data_path:
        return Path(data_path)
    candidates = sorted(p for p in Path('.').rglob('*.csv') if 'artifacts' not in p.parts and not p.name.startswith('.'))
    return candidates[0] if candidates else None

dataset_path = find_dataset(DATA_PATH)
if dataset_path is None or not dataset_path.exists():
    df = make_demo_car_dataset()
    print('CSV не найден. Используется демонстрационный датасет.')
else:
    df = pd.read_csv(dataset_path)
    print('Загружен файл:', dataset_path)

print('Размер данных:', df.shape)
display(df.head())

## 3. Первичная проверка

Проверяем типы данных, пропуски, дубликаты и базовую статистику. Это помогает обнаружить технические колонки, пустую целевую переменную и проблемы качества данных.

In [ ]:
display(df.info())
display(df.describe(include='all').T)

missing = df.isna().mean().mul(100).sort_values(ascending=False).rename('missing_percent').to_frame()
display(missing.head(20))
print('Дубликаты строк:', df.duplicated().sum())

plt.figure(figsize=(10, 4))
missing_to_plot = missing[missing['missing_percent'] > 0].head(25)
if len(missing_to_plot):
    sns.barplot(data=missing_to_plot.reset_index(), x='missing_percent', y='index', color='#4C78A8')
    plt.xlabel('Пропуски, %')
    plt.ylabel('Признак')
    plt.title('Доля пропусков по признакам')
else:
    plt.text(0.5, 0.5, 'Пропусков нет', ha='center', va='center', fontsize=14)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Подготовка целевой переменной

Целевая переменная - цена автомобиля. После выбора цели удаляем строки без цены, дубликаты и технические идентификаторы.

In [ ]:
TARGET_CANDIDATES = ['price', 'Price', 'PRICE', 'selling_price', 'Selling_Price', 'selling price', 'target', 'y', 'цена', 'Цена', 'стоимость', 'Стоимость']
target_col = next((col for col in TARGET_CANDIDATES if col in df.columns), None)
if target_col is None:
    raise ValueError('Не удалось найти колонку с ценой. Укажите target_col вручную.')

data = df.copy().drop_duplicates()
data[target_col] = pd.to_numeric(data[target_col], errors='coerce')
data = data.dropna(subset=[target_col])
data = data[data[target_col] > 0]

id_like_cols = [col for col in data.columns if col != target_col and (col.lower() in {'id', 'car_id', 'vin', 'url', 'link'} or col.lower().endswith('_id'))]
if id_like_cols:
    data = data.drop(columns=id_like_cols)
    print('Удалены технические признаки:', id_like_cols)

print('Целевая переменная:', target_col)
print('Размер после очистки:', data.shape)
display(data[[target_col]].describe().T)

## 5. EDA: цена, корреляции и категории

Распределение цены часто скошено вправо: дорогих машин мало, но они сильно влияют на среднее. Корреляции помогают быстро увидеть линейные связи, а группировки по категориям показывают различия между брендами, типами топлива и кузовами.

In [ ]:
X = data.drop(columns=[target_col])
y = data[target_col]
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=35, kde=True, ax=axes[0], color='#59A14F')
axes[0].set_title('Распределение цены')
sns.histplot(np.log1p(y), bins=35, kde=True, ax=axes[1], color='#F28E2B')
axes[1].set_title('Распределение log1p(цены)')
plt.tight_layout()
plt.show()

if numeric_features:
    corr = data[numeric_features + [target_col]].corr(numeric_only=True)
    plt.figure(figsize=(min(12, 1 + .65 * len(corr.columns)), min(9, 1 + .55 * len(corr.columns))))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, linewidths=.5)
    plt.title('Корреляции числовых признаков')
    plt.tight_layout()
    plt.show()

for col in categorical_features[:6]:
    stats = data.groupby(col, dropna=False)[target_col].agg(['count', 'median']).sort_values('median', ascending=False).head(12).reset_index()
    plt.figure(figsize=(10, 4))
    sns.barplot(data=stats, y=col, x='median', color='#E15759')
    plt.title(f'Медианная цена по категориям: {col}')
    plt.xlabel('Медианная цена')
    plt.tight_layout()
    plt.show()
    display(stats)

print('Числовые признаки:', numeric_features)
print('Категориальные признаки:', categorical_features)

### Промежуточные выводы

Обычно свежий год выпуска и высокая мощность повышают цену, большой пробег снижает ее, а бренд, тип топлива и коробка передач дают заметные различия между группами. Но финальный вывод о важности признаков лучше делать после обучения модели, потому что корреляция показывает только линейную связь.

## 6. Baseline и финальная модель

Сравниваем `RandomForestRegressor` с простым baseline, который предсказывает медианную цену. Если финальная модель не выигрывает у baseline, значит признаки или подготовка данных требуют пересмотра.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)], remainder='drop')

baseline = Pipeline(steps=[('preprocessor', preprocessor), ('model', DummyRegressor(strategy='median'))])
model = Pipeline(steps=[('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=400, min_samples_leaf=2, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1))])

baseline.fit(X_train, y_train)
model.fit(X_train, y_train)

def regression_report(name, fitted_model, X_part, y_part):
    pred = fitted_model.predict(X_part)
    return {'model': name, 'MAE': mean_absolute_error(y_part, pred), 'RMSE': np.sqrt(mean_squared_error(y_part, pred)), 'R2': r2_score(y_part, pred)}

metrics = pd.DataFrame([regression_report('Baseline median', baseline, X_test, y_test), regression_report('RandomForest', model, X_test, y_test)])
display(metrics.style.format({'MAE': '{:,.0f}', 'RMSE': '{:,.0f}', 'R2': '{:.3f}'}))

cv_scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=5, n_jobs=-1)
print(f'CV MAE: {-cv_scores.mean():,.0f} +/- {cv_scores.std():,.0f}')

## 7. Анализ ошибок

График `факт vs прогноз` показывает, насколько модель попадает в реальные цены. Распределение остатков помогает увидеть систематическое завышение или занижение прогноза.

In [ ]:
y_pred = model.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(x=y_test, y=y_pred, alpha=.55, ax=axes[0], color='#4C78A8')
min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--')
axes[0].set_title('Фактическая цена vs прогноз')
axes[0].set_xlabel('Фактическая цена')
axes[0].set_ylabel('Прогноз')

sns.histplot(residuals, bins=35, kde=True, ax=axes[1], color='#B07AA1')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Распределение ошибок')
axes[1].set_xlabel('Факт - прогноз')
plt.tight_layout()
plt.show()

error_analysis = X_test.copy()
error_analysis[target_col] = y_test
error_analysis['prediction'] = y_pred
error_analysis['abs_error'] = np.abs(error_analysis[target_col] - error_analysis['prediction'])
error_analysis['relative_error_percent'] = error_analysis['abs_error'] / error_analysis[target_col] * 100
display(error_analysis.sort_values('abs_error', ascending=False).head(10))

## 8. Важные и неважные признаки

Используем два подхода: встроенную важность Random Forest и permutation importance. Для категориальных признаков важность one-hot колонок агрегируется обратно к исходному признаку.

In [ ]:
def get_feature_names(preprocessor, numeric_features, categorical_features):
    names = list(numeric_features)
    if categorical_features:
        encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
        names.extend(encoder.get_feature_names_out(categorical_features).tolist())
    return names

def map_encoded_to_original(encoded_name):
    if encoded_name in numeric_features:
        return encoded_name
    for col in categorical_features:
        if encoded_name.startswith(col + '_'):
            return col
    return encoded_name

encoded_feature_names = get_feature_names(model.named_steps['preprocessor'], numeric_features, categorical_features)
encoded_importance = pd.DataFrame({'encoded_feature': encoded_feature_names, 'importance': model.named_steps['model'].feature_importances_}).sort_values('importance', ascending=False)
encoded_importance['original_feature'] = encoded_importance['encoded_feature'].apply(map_encoded_to_original)

grouped_importance = encoded_importance.groupby('original_feature', as_index=False)['importance'].sum().sort_values('importance', ascending=False)
grouped_importance['importance_percent'] = grouped_importance['importance'] / grouped_importance['importance'].sum() * 100

display(grouped_importance)
plt.figure(figsize=(10, max(4, .38 * len(grouped_importance))))
sns.barplot(data=grouped_importance, x='importance_percent', y='original_feature', color='#59A14F')
plt.title('Важность исходных признаков')
plt.xlabel('Доля важности, %')
plt.tight_layout()
plt.show()

perm = permutation_importance(model, X_test, y_test, scoring='neg_mean_absolute_error', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_importance = pd.DataFrame({'feature': X_test.columns, 'importance_mean': perm.importances_mean, 'importance_std': perm.importances_std}).sort_values('importance_mean', ascending=False)
display(perm_importance)
plt.figure(figsize=(10, max(4, .38 * len(perm_importance))))
sns.barplot(data=perm_importance, x='importance_mean', y='feature', color='#F28E2B')
plt.title('Permutation importance')
plt.tight_layout()
plt.show()

threshold = grouped_importance['importance_percent'].median()
important_features = grouped_importance[grouped_importance['importance_percent'] >= threshold]
weak_features = grouped_importance[grouped_importance['importance_percent'] < threshold]
print('Важные признаки:')
display(important_features)
print('Менее важные признаки:')
display(weak_features)

### Вывод по признакам

Важными считаем признаки, которые стабильно находятся наверху в общей важности и permutation importance. Обычно для цены автомобиля это год выпуска, пробег, бренд, мощность, объем двигателя, тип топлива, коробка передач и состояние.

Слабые признаки не обязательно нужно сразу удалять: если они почти нулевые и сложны в поддержке, их можно убрать; если признак бизнесово важен, но модель его не использует, стоит проверить заполненность и способ кодирования.

## 9. Сохранение модели и пример прогноза

Сохраняем весь `Pipeline`, включая preprocessing. Такой файл можно применять к новым строкам без ручного кодирования категорий.

In [ ]:
metadata = {'target_col': target_col, 'numeric_features': numeric_features, 'categorical_features': categorical_features, 'test_metrics': metrics.to_dict(orient='records'), 'important_features': important_features['original_feature'].tolist(), 'weak_features': weak_features['original_feature'].tolist()}
with open(ARTIFACTS_DIR / 'car_price_model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

if joblib is not None:
    joblib.dump(model, ARTIFACTS_DIR / 'car_price_model.joblib')
    print('Модель сохранена:', ARTIFACTS_DIR / 'car_price_model.joblib')
else:
    print('joblib не установлен, модель не сохранена.')

sample_objects = X_test.head(5).copy()
prediction_example = sample_objects.copy()
prediction_example['predicted_price'] = model.predict(sample_objects)
display(prediction_example)

## Итог

Построена готовая модель прогнозирования цены автомобиля. Качество оценивается через MAE, RMSE и R2, результат сравнивается с baseline. Графики ошибок показывают, где модель ошибается сильнее всего, а блок важности признаков отделяет полезные характеристики от слабых кандидатов на удаление или дополнительную обработку.

Следующие улучшения: добавить больше данных о комплектации и состоянии, отдельно обработать выбросы, сравнить Random Forest с CatBoost или LightGBM, подобрать гиперпараметры и проверить качество по ценовым сегментам.